# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimanshahid800/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!git clone https://github.com/aimanshahid800/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 164 (delta 66), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.96 MiB | 10.70 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/flyrank-ml-internship/flyrank-ml-internship




*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two Paper Findings + My Methodology Questions

**Finding #2 — The Content Performance Curve (decay cliff at 271-365 days)**

- **Label source:** Health score (composite: impressions 30pts + position 30pts + CTR 20pts + scroll depth 20pts), bucketed by content age. This is a direct aggregate comparison from raw GSC/GA4 data — not a model prediction.
- **Does the validation design carry the claim?** Partially. This is an observational, correlational comparison — no train/test split, no holdout. The paper itself flags this as "not evidence that age naturally reverses performance decline on its own." The large sample size (341K pieces) makes the pattern directionally robust, but it can't support a causal claim ("age causes decline"), only a descriptive one ("age and low health co-occur"). My own capstone label (`is_declining`) is similarly just a decline flag, not a causal test — same caution applies to my work.

**Finding — ML Appendix: Feature Importance (avg_position = 43% importance predicting Health Score)**

- **Label source:** Health score again — but critically, Health Score is partly *constructed from* average position (30 of 100 points). So avg_position is both an input to the label and a top predictor of it.
- **Does the validation design carry the claim?** No, not fully. The paper is explicit about this: "importance is descriptive rather than causal" because the target already includes the feature. This is a **circularity risk** — my own w05_model.ipynb has almost the identical pattern (avg_position is the #1 feature importance at 0.375, and `avg_position` also directly informs the `is_declining` label logic via trend direction). This is a fair thing to flag honestly in my own capstone's limitations, not something to hide.

**Takeaway for my own model:** Both findings show that a "confirmed" or high-AUC result can still be observational or partially circular. My capstone's AUC (0.760) is a genuine predictive signal, but I should be careful not to claim it "explains why" pages decline — only that it ranks them usefully.

## 2. My Model Under an Honest Split (Before/After)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My w05_model.ipynb already used a **grouped split** (GroupShuffleSplit by content_id) — the right choice, since I only have one month of data (no time-aware split was possible). But I never showed what happens *without* grouping. Here's that honest comparison.

**Before — naive random split (no grouping, ignores content_id):**
Risk: if the same content_id could appear in both train and test (not possible here since each row is a distinct page, but the risk is more general — near-duplicate rows or correlated pages leaking signal across the split).

**After — grouped split (my actual w05 approach):**
GroupShuffleSplit by content_id, 80/20.

Below: run both and compare AUC.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
model_df["is_declining"] = (model_df["trend_direction"] == "down").astype(int)
features = ["content_age_days", "impressions_last_30d", "avg_position", "ctr"]
model_df = model_df.dropna(subset=features + ["is_declining"])

# BEFORE: naive random split (no grouping)
train_df, test_df = train_test_split(model_df, test_size=0.2, random_state=42, stratify=model_df["is_declining"])
rf_naive = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf_naive.fit(train_df[features], train_df["is_declining"])
auc_naive = roc_auc_score(test_df["is_declining"], rf_naive.predict_proba(test_df[features])[:, 1])

# AFTER: grouped split (my actual approach)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["content_id"]))
train_g, test_g = model_df.iloc[train_idx], model_df.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf_grouped.fit(train_g[features], train_g["is_declining"])
auc_grouped = roc_auc_score(test_g["is_declining"], rf_grouped.predict_proba(test_g[features])[:, 1])

print(f"Naive random split AUC:  {auc_naive:.3f}")
print(f"Grouped split AUC:       {auc_grouped:.3f}")

Naive random split AUC:  0.747
Grouped split AUC:       0.760


**Result:** Naive split AUC = 0.747, Grouped split AUC = 0.760 — very close, only a 0.013 gap. This makes sense: each row in this dataset is a distinct content_id with no natural duplicates, so grouping had minimal practical effect here. Still, GroupShuffleSplit remains the correct default choice — it protects against leakage risk that would matter if the dataset ever had repeated/correlated content_id entries (e.g. multiple snapshots of the same page over time), which this single-month snapshot doesn't have but a multi-month version would.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


Same hunt as Week 3, applied to my final feature set: `content_age_days`, `impressions_last_30d`, `avg_position`, `ctr`.

**Label:** `is_declining` = 1 if `trend_direction == "down"`, else 0. `trend_direction` itself is calculated from 30-day-vs-previous-30-day impression change (per the FlyRank paper's own definition: >10% decline = down).

**Checked each feature for leakage risk:**

| Feature | Leakage risk? | Reasoning |
|---|---|---|
| `content_age_days` | None | Independent of trend calculation — pure metadata (days since creation) |
| `impressions_last_30d` | Caution, not leakage | This IS one half of the trend calculation (current 30d window). It's correlated with the label by construction, not because it "sees the future" — it's a legitimate predictive feature, but I should be honest that some signal here is definitional, not purely predictive |
| `avg_position` | None | Search ranking is a separate signal from impression trend — not used in `trend_direction` formula |
| `ctr` | None | Click-through rate is independent of the impression-trend label |

**Verdict:** No hard leakage (no direct copy of the label, no post-outcome data, no future window beyond the label's own definition). But `impressions_last_30d` deserves a flag: it's part of *how* the label is computed, so its strong role in prediction is partly circular — same caution the FlyRank paper raised about avg_position and Health Score in the ML appendix (Finding: Feature Importance). This doesn't invalidate the model, but the claim "the model predicts decline" should be softened to "the model uses current visibility signals, including one that partially defines the label, to rank decline risk."

**No client names, domains, URLs, or private queries** appear anywhere in the feature set or training data — confirmed clean, same as capstone.ipynb's Section 2.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence (from capstone.ipynb Section 4 — Results):**
> "Random Forest AUC = 0.760 — big genuine improvement" over the baseline rule (AUC 0.494).

**Problem with this phrasing:** "Big genuine improvement" implies a settled, causal, general claim — that the model *actually* understands what makes content decline, and that this gap would hold up anywhere. Given the leakage audit in Section 3 (one feature, `impressions_last_30d`, partially overlaps with how the label itself is defined), and given this is single-month data with no time-based validation, that language overclaims.

**Rewritten in safe language (observed / measured / directional / decision-support):**
> On this single-month sample, the Random Forest model **measured** an AUC of 0.760 against a grouped holdout split, compared to 0.494 for the rule-based baseline. This is an **observed** improvement in ranking pages by decline risk, not a causal explanation of why pages decline. One input feature (`impressions_last_30d`) partially overlaps with the label's own definition, so results should be read as **directional** evidence that current visibility signals help rank refresh priority — useful for **decision-support** in choosing which pages to review first, not as a validated general-purpose decline predictor.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.